In [3]:
import numpy as np
from scipy.integrate import solve_ivp
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display

def lorenz(t, X, sigma, rho, beta): # Function
    x, y, z = X
    return [
        sigma * (y - x),
        x * (rho - z) - y,
        x * y - beta * z]

current_fig = None
def solve_lorenz(sigma, rho, beta, x0=(0.1,0.1,0.1), tmax=100, dt=0.01): # Start at x,y,z=0.1 to not make everything 0
    t = np.arange(0, tmax, dt)
    sol = solve_ivp(
        lorenz,
        (0, tmax),
        x0,
        args=(sigma, rho, beta),
        t_eval=t,
        rtol=1e-8,
        atol=1e-10)
    x, y, z = sol.y
    return x, y, z

def build_frames(x, y, z, x_error = None, y_error = None, z_error = None, step=5):
    frames = []
    for k in range(0, len(x), step):
        data=[
            go.Scatter3d( # Trajectory
                x=x[:k+1], y=y[:k+1], z=z[:k+1],
                mode="lines",
                line=dict(width=2, color = "#EB5E00")),
            go.Scatter3d( # Moving Point
                x=[x[k]], y=[y[k]], z=[z[k]],
                mode="markers",
                marker=dict(size=5, color = "#3731A9")), 
            go.Scatter3d( # Trajectory with Error
                x=x_error[:k+1], y=y_error[:k+1], z=z_error[:k+1],
                mode="lines",
                line=dict(width=2, color = "#48FF00")),
            go.Scatter3d( # Moving Point with Error
                x=[x_error[k]], y=[y_error[k]], z=[z_error[k]],
                mode="markers",
                marker=dict(size=5, color = "#C51B1B"))]
        frames.append(go.Frame(data=data))
    return frames

eps = 1e-4
def make_figure(sigma, rho, beta):
    x, y, z = solve_lorenz(sigma, rho, beta)
    x_error, y_error, z_error = solve_lorenz(sigma, rho, beta, x0=(0.1+eps, 0.1+eps, 0.1+eps))

    x = x.astype(np.float32)
    y = y.astype(np.float32)
    z = z.astype(np.float32)
    x_error = x_error.astype(np.float32)
    y_error = y_error.astype(np.float32)
    z_error = z_error.astype(np.float32)

    frames = build_frames(x, y, z, x_error, y_error, z_error)
    data = [
        go.Scatter3d(x=[x[0]],y=[y[0]],z=[z[0]], mode = "lines", visible=True),
        go.Scatter3d(x=[x[0]],y=[y[0]],z=[z[0]], mode = "markers", visible=True),
        go.Scatter3d(x=[x_error[0]],y=[y_error[0]],z=[z_error[0]], mode = "lines", visible=False),
        go.Scatter3d(x=[x_error[0]],y=[y_error[0]],z=[z_error[0]], mode = "markers", visible=False)]

    fig = go.Figure(data=data, frames=frames)
    fig.update_layout( # UI Interface with full interaction (thanks to Plotly)
        scene=dict(
            xaxis_title="x",
            yaxis_title="y",
            zaxis_title="z"),
        updatemenus=[dict(
            type="buttons",
        showactive=False,
        buttons=[
            dict(
                label="Play",
                method="animate",
                args=[None, dict(
                    frame=dict(duration=30, redraw=True),
                    transition=dict(duration=0),
                    fromcurrent=True)]),
            dict(
                label="Pause",
                method="animate",
                args=[[None], dict(
                    frame=dict(duration=0),
                    mode="immediate")]),
            dict(
                label="Initiate Perturbation",
                method="update",
                args=[{"visible": [True, True, False, False]}],   # OFF state
                args2=[{"visible": [True, True, True, True]}],)])],    # ON state
    margin=dict(l=0, r=0, b=0, t=100))
    return fig

sigma_slider = widgets.FloatSlider(value=10, min = 0, max = 30, step = 0.5, description = "sigma") # Interactive Sliders
rho_slider   = widgets.FloatSlider(value = 28, min = 0, max = 60,  step = 1, description = "rho")
beta_slider  = widgets.FloatSlider(value = 8/3, min = 0, max = 10, step = 0.1, description = "beta")
update_button = widgets.Button(description="Update")

output = widgets.Output()
def update_plot(_):
    global current_fig
    with output:
        output.clear_output(wait=True)
        current_fig = make_figure(
            sigma_slider.value,
            rho_slider.value,
            beta_slider.value,)
        current_fig.show()

update_button.on_click(update_plot)
update_button.on_click(update_plot)

display(
    widgets.VBox([
        sigma_slider,
        rho_slider,
        beta_slider,
        update_button,
        output]))

update_plot(None) # Load empty plot to begin the Script
current_fig.write_html("Classical_Lorenz_Attractor_3D_Animation.html", include_plotlyjs='cdn', full_html=True) # Save as an Open File